# Computable Systemhood v0.4 — Two-Qubit Numerical Lab

Reproducible toy simulation for Version 2. It does **not** demonstrate additional physical time dimensions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
I2=np.eye(2,dtype=complex)
sx=np.array([[0,1],[1,0]],dtype=complex)
sz=np.array([[1,0],[0,-1]],dtype=complex)
ket00=np.array([1,0,0,0],dtype=complex)
bell=np.array([1,0,0,1],dtype=complex)/np.sqrt(2)
def density(psi): return np.outer(psi,psi.conj())
def entropy(r):
    v=np.linalg.eigvalsh((r+r.conj().T)/2); v=v[v>1e-12]
    return float(-np.sum(v*np.log2(v)))
def ptrace(r,keep):
    R=r.reshape(2,2,2,2)
    return np.trace(R,axis1=1,axis2=3) if keep==0 else np.trace(R,axis1=0,axis2=2)
def C(r): return (entropy(ptrace(r,0))+entropy(ptrace(r,1))-entropy(r))/2
def fidelity_pure(r,psi): return float(np.real(np.vdot(psi,r@psi)))
def Uxx(t,J=1):
    XX=np.kron(sx,sx)
    return np.cos(J*t)*np.eye(4)-1j*np.sin(J*t)*XX
def dephase(r,p):
    ZI=np.kron(sz,I2)
    return (1-p)*r+p*ZI@r@ZI
def autonomy(J,g,J0=.1): return abs(J)/(abs(J)+abs(g)+J0)


## Internal interaction: C(t) and initial-state retention


In [ ]:
ts=np.linspace(0,np.pi,301); cs=[]; fs=[]
for t in ts:
    psi=Uxx(t)@ket00; r=density(psi)
    cs.append(C(r)); fs.append(fidelity_pure(r,ket00))
i=int(np.argmax(cs))
print('max C=',cs[i],' at t=',ts[i],' expected pi/4=',np.pi/4)
plt.figure(figsize=(8,4.5)); plt.plot(ts,cs,label='C(t)'); plt.plot(ts,fs,label='fidelity to |00>'); plt.legend(); plt.xlabel('t (J=1)'); plt.grid(alpha=.25); plt.show()


Low fidelity to the initial state does not mean loss of system identity. Predictive persistence must compare actual future state with an effective model's predicted future state.


## Bell state under phase-flip dephasing


In [ ]:
ps=np.linspace(0,.5,201); rb=density(bell); cd=[]; fb=[]
for p in ps:
    rp=dephase(rb,p); cd.append(C(rp)); fb.append(fidelity_pure(rp,bell))
print('p=0:',cd[0],fb[0]); print('p=.5:',cd[-1],fb[-1])
plt.figure(figsize=(8,4.5)); plt.plot(ps,cd,label='C(p)'); plt.plot(ps,fb,label='Bell fidelity'); plt.legend(); plt.xlabel('phase-flip p'); plt.grid(alpha=.25); plt.show()


## Autonomy scan


In [ ]:
ratios=np.logspace(-2,2,250); av=[autonomy(r,1,.1) for r in ratios]
plt.figure(figsize=(8,4.5)); plt.semilogx(ratios,av); plt.xlabel('J/g (g=1)'); plt.ylabel('A_J'); plt.grid(alpha=.25); plt.show()
for J,g in [(0,0),(1,0),(1,1),(.1,1)]: print(J,g,autonomy(J,g,.1))


## Minimal comparison and scientific boundary
Bell correlation can be maximal while continuing internal interaction is zero. Thus correlation, autonomy and persistence are distinct diagnostics. None of these results is evidence for a new time variable. A later theory must derive a non-reparameterizable relational clock residual relative to GR + standard quantum dynamics.
